## ⚙️ Environment Setup (run once)

Most AgentCore services (Gateway, Memory, Runtime, Registry) require
recent versions of `boto3`. This cell installs/updates everything the
workshop needs.

> ⚠️ **After installing, restart the kernel** (Kernel → Restart) and re-run the
> notebooks. You only need to do this **once** per JupyterLab session.

In [ ]:
%pip install --quiet --upgrade \
    boto3 botocore \
    bedrock-agentcore bedrock-agentcore-starter-toolkit \
    mcp PyJWT requests
print("✓ Dependencies installed/updated.")
print("⚠️  If this is the first time in this session, restart the kernel now")
print("   (Kernel → Restart Kernel) e re-run the notebooks.")

# Lab 01.1 — Create Cognito User Pool with Groups

## Overview

In this lab you will create the identity foundation for the workshop:

- An **Amazon Cognito User Pool** (authentication)
- 3 **groups** (`operators`, `managers`, `governance`) that become the `cognito:groups` claim in the JWT
- 2 **test users** (Ana as operator and Carlos as manager)

The `cognito:groups` claim is what the Cedar Policy Engine will use in Lab 03 to
decide who can call which tools on the Gateway.

> 💡 **Mental model:** Cognito is the *single source of truth* about who the
> user is. All other AgentCore components (Gateway, Runtime, Cedar)
> simply trust the JWT issued by Cognito.

## Tutorial Details

| Information | Details |
|---|---|
| Tutorial type | Interactive |
| AgentCore components | Identity (via [Amazon Cognito](https://docs.aws.amazon.com/cognito/)) |
| Framework | — |
| Complexity | Easy |
| SDK | boto3 |
| Estimated time | 5 minutes |

## Prerequisites

- AWS account with configured credentials (`aws configure`)
- Python 3.10+ with `pip install -r requirements.txt`
- IAM permission to create Cognito User Pools

> ℹ️ Lab 01.1 (this one) and Lab 01.2 (next) **do not depend** on other labs.

## Setup

In [ ]:
import os
import sys
import json

# Allows importing shared/utils/ and the local utils.py for this lab
sys.path.insert(0, "..")  # workshop root

from shared.utils.config import load_config, save_config, get_region
from utils import create_user_pool_with_groups, DEFAULT_GROUPS, DEFAULT_USERS

# Loads config.env (creates from .example if it doesn't exist yet)
cfg = load_config()
region = get_region()
print(f"Region: {region}")

## Step 1: View the configuration we will apply

The default groups and users come from [`utils.py`](./utils.py). You can
customize them by passing `groups=` and `users=` to `create_user_pool_with_groups()`.

In [ ]:
print("Groups to be created:")
for g in DEFAULT_GROUPS:
    print(f"  • {g['name']:12s} — {g['description']}")

print("\nUsers to be created:")
for u in DEFAULT_USERS:
    groups_str = ', '.join(u['groups'])
    print(f"  • {u['email']:35s} → groups: {groups_str}")

## Step 2: Create User Pool, groups, and users

The function `create_user_pool_with_groups()` is **idempotent**: running it twice won't fail.

> ⚠️ **Workshop password.** We are using a fixed permanent password
> (`Workshop@2025!`) for simplicity. **Never do this in production.**

In [ ]:
result = create_user_pool_with_groups(
    pool_name="workshop-ai-agents-pool",
    user_password="Workshop@2025!",
    region=region,
)
print(json.dumps(result, indent=2, default=str))

## Step 3: Persist IDs to config.env

In [ ]:
save_config({
    "COGNITO_USER_POOL_ID": result["pool_id"],
    "COGNITO_USER_POOL_ARN": result["pool_arn"],
    "COGNITO_DISCOVERY_URL": result["discovery_url"],
})

# These IDs will be available in the next labs via load_config().

## ✅ Validation

The OpenID Connect (OIDC) discovery URL should respond with the pool
configuration — including the JWKS URI that will be used by the Gateway in Lab 02 to
validate JWTs.

In [ ]:
import urllib.request

with urllib.request.urlopen(result["discovery_url"]) as resp:
    oidc = json.loads(resp.read())

print("OIDC discovery OK")
print(f"  issuer:                {oidc['issuer']}")
print(f"  jwks_uri:              {oidc['jwks_uri']}")
print(f"  token_endpoint:        {oidc['token_endpoint']}")
print(f"  authorization_endpoint: {oidc['authorization_endpoint']}")

## 🎓 What you learned

- How to create a Cognito User Pool programmatically
- How to organize users into groups (which become claims in the JWT)
- The OpenID Connect discovery endpoint that will be used by the Gateway

## Cleanup

If you want to remove **only** what this notebook created:

```python
from utils import cleanup_user_pool
cleanup_user_pool("workshop-ai-agents-pool", region=region)
```

For full workshop teardown:

```bash
python -m shared.utils.cleanup --identity
```

## Next

➡️ Next notebook: **[01.2 — customClaims and allowedScopes](./02-customclaims-and-allowedscopes.ipynb)**

We will create the OAuth app client, examine the JWT structure, and configure the
parameters that the Gateway will need in Lab 02.